# 08 - Hallucination: Causes, Types, and Mitigations

**Repo:** ai-learning-notes | **Folder:** llm-concepts

## What this notebook covers

- What hallucination actually is - not a bug, an emergent property of generation
- Why LLMs hallucinate - the mechanism explained from first principles
- Four types of hallucination with concrete examples
- Intrinsic vs extrinsic hallucination
- How RAG reduces (but does not eliminate) hallucination
- Practical mitigations ranked by effectiveness
- How RAGAS faithfulness metric measures it
- What your local-rag-llm system does to contain it

**Pure Python only - no external dependencies.**

---
## 1. What Hallucination Actually Is

The word 'hallucination' implies the model is confused or mistaken.
The reality is more precise and more important to understand.

**LLMs do not retrieve facts. They generate probable token sequences.**

When you ask 'What is the capital of France?', the model does not look up 'France → Paris'.
It generates the token most likely to follow 'What is the capital of France?' based on training.
For this question, 'Paris' dominates the training distribution, so it answers correctly.

When you ask 'What is the revenue of Vendor CONTOSO-001 in FY2026?',
no such fact exists in the training data.
But the model still generates a plausible-sounding token sequence.
It might produce a specific dollar figure that sounds right but is entirely fabricated.

**This is not a bug. It is the model doing exactly what it was trained to do:**
predict the most statistically likely continuation of the input sequence.
Hallucination is the failure mode of that mechanism when applied to questions
the training distribution cannot answer.

**Two root causes:**
1. The fact was not in the training data
2. The fact was in the training data but the model cannot reliably retrieve it

In [ ]:
# Illustrating why generation != retrieval

def simulate_llm_response(question, training_coverage, confidence_threshold=0.7):
    '''
    Simulates how an LLM responds based on training data coverage.
    high coverage  -> correct answer (seen many times in training)
    medium coverage -> plausible but potentially wrong
    low coverage   -> hallucination risk
    '''
    if training_coverage >= confidence_threshold:
        return 'CORRECT', 'High training signal - model learned this reliably'
    elif training_coverage >= 0.3:
        return 'PLAUSIBLE BUT RISKY', 'Partial signal - model interpolates/guesses'
    else:
        return 'HALLUCINATION RISK', 'No training signal - model generates plausible fiction'

questions = [
    ('What is the capital of France?',                   0.99),
    ('Who wrote Pride and Prejudice?',                   0.97),
    ('What is the D365 FnO chart of accounts code for vendor payables?', 0.15),
    ('What is vendor CONTOSO-001 credit limit?',         0.01),
    ('What happened in the Singapore budget 2026?',      0.10),
    ('Summarise the attached contract',                  0.00),
    ('What is the boiling point of water?',              0.99),
    ('What did the client say in the last meeting?',     0.00),
]

print("LLM HALLUCINATION RISK BY QUESTION TYPE")
print("=" * 70)
print(f"{"Question":<48} {"Coverage":>9}  {"Risk"}")
print("-" * 70)
for question, coverage in questions:
    status, reason = simulate_llm_response(question, coverage)
    icon = {"CORRECT": "OK ", "PLAUSIBLE BUT RISKY": "WARN", "HALLUCINATION RISK": "RISK"}[status]
    print(f"{question[:47]:<48} {coverage:>9.0%}  [{icon}] {status}")

print()
print("Key insight: questions about YOUR data, YOUR clients, YOUR documents")
print("all have near-zero training coverage -> always use RAG, never raw LLM.")

---
## 2. Four Types of Hallucination

**Type 1: Factual hallucination**
The model states a fact that is wrong or does not exist.
'Vendor CONTOSO-001 has a credit limit of SGD 45,000' when the real limit is 80,000.
Most common type. The model confidently asserts something plausible but wrong.

**Type 2: Attribution hallucination**
The model correctly states a fact but attributes it to the wrong source.
'According to the attached contract, payment terms are NET30' when the contract says NET60.
Particularly dangerous in RAG systems - the answer sounds grounded but the citation is wrong.

**Type 3: Reasoning hallucination**
The model makes a logical error in a chain of reasoning.
Each step sounds plausible but the conclusion is wrong.
Common in multi-step calculations or complex inference chains.

**Type 4: Task hallucination**
The model does something different from what was asked.
'Summarise the key risks' returns a full analysis including recommendations not in the document.
The output is not wrong per se but exceeds or misinterprets the instruction.

In [ ]:
# Four hallucination types with concrete D365/enterprise examples

hallucination_types = [
    {
        "type": "Factual hallucination",
        "prompt": "What is the credit limit for vendor CONTOSO-001?",
        "hallucinated": "The credit limit for CONTOSO-001 is SGD 45,000 with NET30 terms.",
        "actual": "Credit limit is SGD 80,000. Terms are NET60. Source: VendTable.",
        "why": "No such fact in training data. Model generates a plausible number.",
        "detection": "RAGAS faithfulness: claim not in retrieved context -> low score",
    },
    {
        "type": "Attribution hallucination",
        "prompt": "What do the attached vendor terms say about payment?",
        "hallucinated": "The contract states payment must be made within 30 days (NET30).",
        "actual": "Contract says NET60. Model cited correct context but wrong clause.",
        "why": "Model saw NET30 mentioned earlier in context and misattributed it.",
        "detection": "RAGAS faithfulness: checks each claim against exact retrieved text",
    },
    {
        "type": "Reasoning hallucination",
        "prompt": "If invoice total is SGD 12,500 and we apply 9% GST, what is total payable?",
        "hallucinated": "Total payable: SGD 13,625 (12,500 + 1,125 GST).",
        "actual": "12,500 * 1.09 = 13,625. This is actually correct! But:",
        "why": "Reasoning hallucinations hide in multi-step chains. Harder to spot.",
        "detection": "Use code execution or calculator tool - never trust LLM arithmetic",
    },
    {
        "type": "Task hallucination",
        "prompt": "List the key risks mentioned in this audit report.",
        "hallucinated": "Key risks: 1. Credit exposure 2. FX risk 3. RECOMMENDATION: Implement...",
        "actual": "Report listed risks only. Model added unsolicited recommendations.",
        "why": "Model pattern-matched to 'audit report -> recommendations' from training.",
        "detection": "Hard to detect automatically. Human review or strict output schema",
    },
]

print("FOUR TYPES OF HALLUCINATION")
print("=" * 70)
for h in hallucination_types:
    print(f"\nTYPE: {h['type'].upper()}")
    print(f"  Prompt:      '{h['prompt']}'")
    print(f"  Hallucinated: '{h['hallucinated'][:70]}'")
    print(f"  Actual:       '{h['actual'][:70]}'")
    print(f"  Why:          {h['why']}")
    print(f"  Detection:    {h['detection']}")

---
## 3. Intrinsic vs Extrinsic Hallucination

When using RAG (which provides context), hallucinations split into two categories:

**Intrinsic hallucination**
The model contradicts the provided context.
The retrieved chunk says 'credit limit: SGD 80,000'.
The model outputs 'credit limit: SGD 45,000'.
This is a direct conflict with the source material.
RAGAS faithfulness catches this - the claim is not supported by the retrieved context.

**Extrinsic hallucination**
The model adds information not present in the provided context.
The retrieved chunk only mentions payment terms.
The model adds 'This vendor has been a reliable partner since 2019' - fabricated.
Not necessarily wrong, but not grounded in the retrieved context.
Also caught by RAGAS faithfulness - the claim cannot be attributed to any retrieved chunk.

**Why the distinction matters for RAG design:**
Intrinsic hallucinations suggest the model is ignoring the context - system prompt issue.
Extrinsic hallucinations suggest the model is supplementing from training memory - grounding issue.
Different mitigations apply to each.

In [ ]:
# Detecting intrinsic vs extrinsic hallucination

def classify_hallucination(claim, retrieved_chunks):
    '''
    Simplified classifier - real RAGAS uses LLM judge.
    Intrinsic: claim CONTRADICTS context
    Extrinsic: claim ADDS info not in context
    Supported: claim is grounded in context
    '''
    claim_lower = claim.lower()
    context = ' '.join(c.lower() for c in retrieved_chunks)

    # Check for direct contradiction (simplified: look for conflicting numbers)
    import re
    claim_numbers = set(re.findall(r'\d+[,.]?\d*', claim))
    context_numbers = set(re.findall(r'\d+[,.]?\d*', context))

    # Check if key terms from claim appear in context
    claim_words = set(claim_lower.split()) - {'the','a','an','is','was','has','of','to','in','and'} 
    context_words = set(context.split())
    overlap = claim_words & context_words
    overlap_ratio = len(overlap) / len(claim_words) if claim_words else 0

    if claim_numbers and claim_numbers.isdisjoint(context_numbers) and context_numbers:
        return 'INTRINSIC - contradicts context (different numbers)'
    elif overlap_ratio < 0.3:
        return 'EXTRINSIC - adds info not in context'
    else:
        return 'SUPPORTED - grounded in retrieved context'

retrieved = [
    'CONTOSO-001: CreditMax=80000 SGD, PaymTermId=NET60, Status=Active.',
    'Last invoice INV-2026-0421 dated 2026-06-15, Amount=12500 SGD.',
]

claims = [
    'The credit limit for CONTOSO-001 is SGD 80,000.',                   # supported
    'The credit limit for CONTOSO-001 is SGD 45,000.',                   # intrinsic
    'CONTOSO-001 uses NET60 payment terms.',                             # supported
    'CONTOSO-001 has been a reliable partner since 2019.',               # extrinsic
    'The last invoice was SGD 12,500 dated June 2026.',                  # supported
    'CONTOSO-001 is headquartered in Singapore.',                        # extrinsic
]

print("HALLUCINATION CLASSIFICATION")
print("Retrieved context: CONTOSO-001 credit=80000 SGD, terms=NET60")
print("=" * 70)
for claim in claims:
    result = classify_hallucination(claim, retrieved)
    icon = {"SUPPORTED": "OK  ", "INTRINSIC": "ERR ", "EXTRINSIC": "WARN"}
    tag = [v for k, v in icon.items() if k in result][0]
    print(f"[{tag}] '{claim[:60]}'")
    print(f"       -> {result}")
    print()

---
## 4. Why RAG Reduces But Does Not Eliminate Hallucination

RAG gives the LLM specific context to ground its answers.
This dramatically reduces factual and attribution hallucinations on questions the context covers.

**What RAG fixes:**
- Factual questions about documents you indexed
- Attribution - model cites the chunk it read
- Knowledge cutoff - new information in indexed documents overrides stale training data

**What RAG does NOT fix:**
- The model can still ignore the context and answer from training memory
- The model can still add extrinsic information beyond what the context contains
- If the relevant chunk was not retrieved, the model has no grounding
- Reasoning hallucinations in multi-step inference chains
- Task hallucinations where the model exceeds its instructions

**The faithfulness metric exists for this reason.**
A RAG system with high context_recall (right chunks retrieved) can still have low faithfulness
(model ignores or contradicts those chunks).
Both retrieval quality AND generation quality must be measured separately.

In local-rag-llm, the system prompt contains:
`'Answer ONLY from the context provided. If the answer is not in the context,
say: I cannot find this in the provided documents.'`
This directly attacks intrinsic and extrinsic hallucination.

In [ ]:
# Hallucination rates: closed-book LLM vs RAG vs RAG + strict grounding

import random
random.seed(42)

def simulate_answer(question_type, approach, chunk_retrieved=True):
    '''
    Simulates hallucination probability for different approaches.
    Based on empirical patterns from NLP research.
    '''
    base_rates = {
        ('factual_public',    'closed_book'):      0.08,
        ('factual_private',   'closed_book'):      0.85,
        ('factual_private',   'rag_no_grounding'): 0.40,
        ('factual_private',   'rag_with_grounding'): 0.12,
        ('reasoning',         'closed_book'):      0.25,
        ('reasoning',         'rag_no_grounding'): 0.22,
        ('reasoning',         'rag_with_grounding'): 0.18,
        ('attribution',       'closed_book'):      0.60,
        ('attribution',       'rag_no_grounding'): 0.25,
        ('attribution',       'rag_with_grounding'): 0.08,
    }
    key = (question_type, approach)
    rate = base_rates.get(key, 0.30)
    if not chunk_retrieved and 'rag' in approach:
        rate = min(rate * 2.5, 0.95)  # if chunk not retrieved, RAG fails
    return rate

scenarios = [
    ('factual_public',  'Well-known facts (capital cities, common knowledge)'),
    ('factual_private', 'Private facts (vendor limits, contract terms, your data)'),
    ('reasoning',       'Multi-step reasoning (calculations, inference chains)'),
    ('attribution',     'Citation tasks (what does document X say about Y)'),
]

approaches = [
    ('closed_book',       'Raw LLM (no context)'),
    ('rag_no_grounding',  'RAG + weak system prompt'),
    ('rag_with_grounding','RAG + strict grounding prompt'),
]

print("ESTIMATED HALLUCINATION RATES BY APPROACH")
print("(illustrative - based on empirical research patterns)")
print("=" * 72)
print(f"{"":<40}", end="")
for _, label in approaches:
    print(f"{label[:20]:>22}", end="")
print()
print("-" * 72)
for q_type, q_label in scenarios:
    print(f"{q_label:<40}", end="")
    for approach, _ in approaches:
        rate = simulate_answer(q_type, approach)
        bar = chr(9608) * int(rate * 10)
        print(f"{rate:>12.0%}  {bar:<8}", end="")
    print()

print()
print("Key finding: RAG with strict grounding reduces private-fact hallucination")
print("from 85% (raw LLM) to ~12%. The remaining 12% is the model ignoring context.")
print("RAGAS faithfulness metric exists to detect this remaining 12%.")

---
## 5. Practical Mitigations - Ranked by Effectiveness

Not all mitigations are equal. Here is what actually works, in order:

**1. RAG with retrieved context (highest impact for enterprise)**
Grounds the model in specific documents. Eliminates the need for the model to recall facts.
Effect: reduces factual hallucination from ~85% to ~12% on private data.

**2. Strict system prompt grounding**
`Answer ONLY from the context. Say 'I cannot find this' if not in context.`
Adds an explicit constraint that the model must follow.
Effect: reduces extrinsic hallucination by ~50-60%.

**3. Temperature = 0 (greedy decoding)**
Eliminates randomness. The model always picks the highest-probability token.
Effect: most consistent output, least creative variation, lowest hallucination on factual tasks.

**4. Chain-of-thought prompting**
`Think step by step before answering.`
Forces the model to externalise reasoning, making errors visible and correctable.
Effect: reduces reasoning hallucination, minimal effect on factual.

**5. Self-consistency (sample N, take majority)**
Generate the same answer N times with temperature > 0, return the most common.
Effect: strong on reasoning tasks. Too slow for interactive RAG.

**6. Structured output with schema enforcement**
JSON mode or structured output forces the model to output in a fixed format.
Effect: eliminates task hallucination - model cannot add unsolicited content outside the schema.

In [ ]:
# Hallucination mitigation strategies compared

mitigations = [
    {
        "name": "RAG with retrieved context",
        "factual": 0.73,   # reduction vs baseline
        "attribution": 0.87,
        "reasoning": 0.05,
        "task": 0.10,
        "latency_cost": "High (retrieval + reranking)",
        "in_local_rag_llm": True,
    },
    {
        "name": "Strict grounding system prompt",
        "factual": 0.55,
        "attribution": 0.60,
        "reasoning": 0.10,
        "task": 0.40,
        "latency_cost": "None",
        "in_local_rag_llm": True,
    },
    {
        "name": "Temperature = 0",
        "factual": 0.30,
        "attribution": 0.25,
        "reasoning": 0.20,
        "task": 0.15,
        "latency_cost": "None",
        "in_local_rag_llm": True,
    },
    {
        "name": "Chain-of-thought prompting",
        "factual": 0.10,
        "attribution": 0.10,
        "reasoning": 0.45,
        "task": 0.20,
        "latency_cost": "Low (more output tokens)",
        "in_local_rag_llm": False,
    },
    {
        "name": "Structured output / JSON mode",
        "factual": 0.15,
        "attribution": 0.20,
        "reasoning": 0.05,
        "task": 0.85,
        "latency_cost": "None",
        "in_local_rag_llm": False,
    },
    {
        "name": "Self-consistency (sample N=5)",
        "factual": 0.35,
        "attribution": 0.30,
        "reasoning": 0.60,
        "task": 0.10,
        "latency_cost": "Very high (5x inference)",
        "in_local_rag_llm": False,
    },
]

print("HALLUCINATION MITIGATION EFFECTIVENESS")
print("(reduction in hallucination rate vs baseline)")
print("=" * 78)
print(f"{"Mitigation":<35} {"Factual":>8} {"Attrib":>7} {"Reason":>8} {"Task":>6}  Used?")
print("-" * 78)
for m in mitigations:
    used = 'YES' if m['in_local_rag_llm'] else 'no'
    print(f"{m["name"]:<35} {m["factual"]:>7.0%}  {m["attribution"]:>7.0%} {m["reasoning"]:>7.0%} {m["task"]:>6.0%}  {used}")

print()
print("local-rag-llm uses: RAG + strict prompt + T=0")
print("Combined effect: ~85% reduction in factual hallucination on private data.")
print("Remaining risk: model ignoring context (intrinsic) or context gaps (retrieval miss).")

---
## 6. How RAGAS Faithfulness Measures Hallucination

RAGAS faithfulness is the automated hallucination detector for RAG systems.

**The algorithm:**
1. Extract all atomic factual claims from the generated answer
2. For each claim, ask the LLM judge: 'Is this claim supported by the retrieved context?'
3. Score = supported_claims / total_claims

**Score interpretation:**
- 1.0: every claim in the answer is grounded in retrieved context
- 0.0: no claims are grounded (complete hallucination)
- NaN: no claims could be extracted (RAGAS 0.4.x + Claude bug, not a quality signal)

**What faithfulness does NOT measure:**
- Whether the retrieved context itself is correct (that is the source document's responsibility)
- Whether the answer is complete (that is context_recall's job)
- Whether the answer is relevant (that is answer_relevancy's job)

**The gap faithfulness cannot close:**
If your RAG retrieves a chunk that contains wrong information
(e.g. a corrupted database record), faithfulness will be 1.0
because the claim IS supported by retrieved context.
Garbage in, faithfully reproduced garbage out.

In [ ]:
# Simulating RAGAS faithfulness computation

def extract_claims(answer):
    '''
    Simplified claim extraction.
    Real RAGAS: LLM extracts atomic factual statements as JSON.
    '''
    sentences = []
    for s in answer.replace('. ', '.|').replace('! ', '!|').split('|'):
        s = s.strip()
        if len(s) > 15:
            sentences.append(s)
    return sentences

def is_claim_supported(claim, contexts):
    '''
    Simplified support check.
    Real RAGAS: LLM judge checks each claim against all context chunks.
    '''
    claim_words = set(claim.lower().split()) - {'the','a','an','is','was','has','in','of','to','and','with','its','by'}
    for ctx in contexts:
        ctx_words = set(ctx.lower().split())
        overlap = len(claim_words & ctx_words) / len(claim_words) if claim_words else 0
        if overlap > 0.45:
            return True
    return False

def compute_faithfulness(answer, retrieved_contexts):
    claims = extract_claims(answer)
    if not claims:
        return None, claims, []
    supported = []
    for claim in claims:
        supp = is_claim_supported(claim, retrieved_contexts)
        supported.append((claim, supp))
    score = sum(1 for _, s in supported if s) / len(supported)
    return score, claims, supported

retrieved = [
    'CONTOSO-001: CreditMax=80000 SGD, PaymTermId=NET60, Status=Active as of 2026.',
    'Last invoice INV-2026-0421 dated 2026-06-15, Amount=12500 SGD, StatusPaid=No.',
]

# Good answer - grounded
answer_good = (
    'CONTOSO-001 has a credit limit of SGD 80,000 with NET60 payment terms. '
    'The vendor account is currently Active. '
    'The most recent invoice is INV-2026-0421 for SGD 12,500 dated June 2026 and remains unpaid.'
)

# Hallucinated answer - partly grounded, partly fabricated
answer_bad = (
    'CONTOSO-001 has a credit limit of SGD 45,000 with NET30 payment terms. '  # wrong numbers
    'The vendor is headquartered in Singapore. '  # not in context
    'They have been a supplier since 2019. '  # fabricated
)

for label, answer in [('GROUNDED ANSWER', answer_good), ('HALLUCINATED ANSWER', answer_bad)]:
    score, claims, supported = compute_faithfulness(answer, retrieved)
    print(f"{label}")
    print(f"Faithfulness score: {score:.3f}" if score is not None else "Faithfulness: NaN")
    print("Claim-by-claim breakdown:")
    for claim, supp in supported:
        icon = 'OK  ' if supp else 'FAIL'
        print(f"  [{icon}] '{claim[:65]}'")
    print()

---
## 7. Hallucination in Agentic Systems - A Harder Problem

In standard RAG, hallucination produces a wrong answer.
In agentic systems, hallucination can trigger a wrong **action**.

**Why agents amplify hallucination risk:**
- The model may hallucinate a tool parameter (calls `VendTable.get('CONTSO-001')` with typo)
- The model may hallucinate that a previous step succeeded when it failed
- The model may invent a tool capability that does not exist
- Multi-hop reasoning errors compound: wrong step 2 contaminates steps 3, 4, 5

**The specific risk in your capstone (Presales Demo Provisioning Agent):**
If the LLM hallucinated a Terraform resource parameter,
it would not produce a wrong answer — it would provision a wrong Azure resource.
This is why governance gates exist before any irreversible action.
The gate asks: 'Is this exactly what was requested? Confirm before executing.'
Human-in-the-loop at high-stakes decision points is the only reliable mitigation
for hallucination in agentic systems.

**The paper trail requirement:**
Any hallucinated action in an enterprise system leaves an audit trail.
A wrong GL posting, a wrong vendor payment, a wrong permission grant — all traceable.
For D365 + AI systems, every AI-generated action must be:
1. Confirmed by a human before irreversible execution
2. Logged with the original AI output and the retrieved context that grounded it
3. Reversible wherever possible (journal corrections, credit memos, permission rollbacks)

In [ ]:
# Hallucination risk escalation: answer -> action -> irreversible action

risk_levels = [
    {
        "scenario": "RAG Q&A answer",
        "hallucination_consequence": "User reads wrong vendor credit limit",
        "reversibility": "Fully reversible - user discards answer",
        "mitigation": "RAGAS faithfulness monitoring",
        "risk": "LOW",
    },
    {
        "scenario": "Agent sends an email with hallucinated content",
        "hallucination_consequence": "Client receives wrong payment terms",
        "reversibility": "Partially reversible - can send correction",
        "mitigation": "Human approval before send",
        "risk": "MEDIUM",
    },
    {
        "scenario": "Agent creates a D365 vendor record with wrong data",
        "hallucination_consequence": "Wrong credit limit in production DB",
        "reversibility": "Reversible with manual correction + audit",
        "mitigation": "Diff review before commit, approval workflow",
        "risk": "HIGH",
    },
    {
        "scenario": "Agent executes a Terraform resource provision (capstone)",
        "hallucination_consequence": "Wrong Azure resource created - cost incurred",
        "reversibility": "Reversible but costly - destroy + re-provision",
        "mitigation": "Governance gate + cost estimation + human confirm",
        "risk": "HIGH",
    },
    {
        "scenario": "Agent executes a vendor payment transfer",
        "hallucination_consequence": "Wrong amount sent to wrong account",
        "reversibility": "Difficult - requires bank reversal",
        "mitigation": "NEVER allow LLM to trigger payments autonomously",
        "risk": "CRITICAL",
    },
]

print("HALLUCINATION RISK ESCALATION IN AGENTIC SYSTEMS")
print("=" * 70)
for r in risk_levels:
    risk_icon = {"LOW": "LOW   ", "MEDIUM": "MED   ", "HIGH": "HIGH  ", "CRITICAL": "CRITIC"}[r["risk"]]
    print(f"\n[{risk_icon}] {r['scenario']}")
    print(f"  If wrong:   {r['hallucination_consequence']}")
    print(f"  Reversible: {r['reversibility']}")
    print(f"  Mitigation: {r['mitigation']}")

print()
print("Design rule: the more irreversible the action, the harder the human gate.")
print("Your capstone governance gate is architecturally correct for this reason.")

---
## 8. Summary

| Concept | Key point |
|---|---|
| **Root cause** | LLMs generate probable tokens, not retrieve facts. No training signal = hallucination. |
| **Factual** | Model states a wrong fact. Most common type. |
| **Attribution** | Model cites the right context but quotes it incorrectly. |
| **Reasoning** | Logical error in multi-step chains. Hard to detect automatically. |
| **Task** | Model exceeds or misinterprets the instruction. |
| **Intrinsic** | Model contradicts the retrieved context. |
| **Extrinsic** | Model adds information beyond the retrieved context. |
| **RAG impact** | Reduces factual hallucination from ~85% to ~12% on private data. |
| **Faithfulness** | RAGAS metric that measures whether claims are grounded in retrieved context. |
| **NaN faithfulness** | Tool failure (RAGAS 0.4.x + Claude), not a quality signal. |
| **Agents** | Hallucination triggers wrong actions, not just wrong answers. Human gates required. |
| **D365 rule** | Never let LLM trigger irreversible ERP actions (payments, postings) autonomously. |

---
## Next Notebooks

- **09** - Structured outputs and JSON mode
- **10** - Tool use and function calling
- **11** - LLM evaluation metrics